# LLM Pipeline Test for TextWorld-Express

This notebook adapts the self-evolving MCTS pipeline to the TextWorld-Express homework game from `hw2_part2.ipynb`.

Workflow:
1. Build a TextWorld game adapter compatible with the MCTS framework
2. Play a baseline MCTS game and collect a trace
3. Run the LLM optimizer on one target MCTS phase
4. Evaluate baseline vs optimized on the anchor task
5. Evaluate baseline vs optimized on a small task suite


## Cell 1: API Setup

In [ ]:
import os
# Set these before running the optimizer.
os.environ.setdefault('API_KEYS', '')
os.environ.setdefault('OPENAI_BASE_URL', 'https://tritonai-api.ucsd.edu')
os.environ.setdefault('MODEL_NAME', 'api-gpt-oss-120b')
print('MODEL_NAME =', os.environ.get('MODEL_NAME'))
print('BASE_URL   =', os.environ.get('OPENAI_BASE_URL'))
print('HAS_KEY    =', bool(os.environ.get('API_KEYS')))


## Cell 2: Imports

In [ ]:
from tools.textworld_llm_pipeline import (
    TextWorldEvalConfig,
    collect_trace_and_sources,
    compare_suite,
    composite_score,
    default_suite,
    format_rows,
    make_engine,
    multi_eval,
    run_single_iteration,
)


## Cell 3: Experiment Parameters

In [ ]:
PHASE = 'simulation'  # try 'selection' later
ITERS = 100
MAX_DEPTH = 50
MAX_STEPS = 50
EVAL_RUNS = 3

ANCHOR = TextWorldEvalConfig(
    game_type='coin',
    game_params='numLocations=5,includeDoors=1,numDistractorItems=0',
    seed=3,
    iterations=ITERS,
    max_rollout_depth=MAX_DEPTH,
    max_steps=MAX_STEPS,
)

SUITE = default_suite(seed=3, iterations=ITERS, max_rollout_depth=MAX_DEPTH, max_steps=MAX_STEPS)
ANCHOR


## Cell 4: Baseline Trace Collection

In [ ]:
trace_result, tool_sources = collect_trace_and_sources(ANCHOR)
print('solved   =', trace_result['solved'])
print('steps    =', trace_result['steps'])
print('returns  =', trace_result['returns'])
print('log_file =', trace_result.get('log_file'))
print('tool phases =', list(tool_sources.keys()))


## Cell 5: Baseline Evaluation on Anchor Task

In [ ]:
base_ret, base_solve, base_steps, base_runs, base_time = multi_eval(ANCHOR, PHASE, fn=None, n=EVAL_RUNS)
print('baseline avg_ret   =', round(base_ret, 4))
print('baseline solve_rate=', round(base_solve, 4))
print('baseline avg_steps =', round(base_steps, 2))
print('baseline composite =', round(composite_score(base_solve, base_ret), 4))
print('baseline eval time =', round(base_time, 2), 's')


## Cell 6: Single LLM Optimization Iteration

In [ ]:
result = run_single_iteration(
    ANCHOR,
    phase=PHASE,
    eval_runs=EVAL_RUNS,
    session_tag=f'textworld_{PHASE}_iter1',
)
best_fn = result.get('function')
print('smoke_test =', result['smoke_test'])
print('error      =', result['error'])
print('installed   =', result.get('installed_path'))
print('baseline_eval  =', result['baseline_eval'])
print('optimized_eval =', result['optimized_eval'])


## Cell 7: Compare on the Anchor Task

In [ ]:
if best_fn is None:
    print('No optimized function was returned.')
else:
    opt_ret, opt_solve, opt_steps, opt_runs, opt_time = multi_eval(ANCHOR, PHASE, fn=best_fn, n=EVAL_RUNS)
    print('optimized avg_ret   =', round(opt_ret, 4))
    print('optimized solve_rate=', round(opt_solve, 4))
    print('optimized avg_steps =', round(opt_steps, 2))
    print('optimized composite =', round(composite_score(opt_solve, opt_ret), 4))
    print('optimized eval time =', round(opt_time, 2), 's')


## Cell 8: Small Suite Comparison

In [ ]:
rows = compare_suite(PHASE, best_fn, SUITE, games_per_config=3)
print(format_rows(rows))


## Notes

- Start with `PHASE='simulation'` before trying `selection`.
- Start with the `coin` anchor task; it is smaller and faster.
- The TextWorld adapter clones by replaying history from the seed, so this notebook is slower than the Sokoban pipeline.
